In [5]:
# Task 2 — Sentiment Analysis for Bank Reviews
# Using Hugging Face DistilBERT model:
# distilbert-base-uncased-finetuned-sst-2-english

# Install first (run once in terminal or notebook)
# pip install transformers torch pandas scipy

import pandas as pd
import torch
from transformers import pipeline
from scipy.special import softmax

# -----------------------------
# 1. Load cleaned dataset
# -----------------------------
df = pd.read_csv("cleaned_reviews.csv")

# -----------------------------
# 2. Load sentiment model
# -----------------------------
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# -----------------------------
# 3. Sentiment function
# -----------------------------
def analyze_sentiment(text):
    
    # Handle empty text
    if pd.isna(text):
        return pd.Series(["neutral", 0.0])
    
    result = classifier(str(text))[0]
    
    label = result["label"].lower()
    score = result["score"]

    # Convert labels
    if label == "positive":
        sentiment = "positive"
    elif label == "negative":
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return pd.Series([sentiment, score])

# -----------------------------
# 4. Apply sentiment analysis
# -----------------------------
df[["sentiment", "confidence"]] = df["review text"].apply(analyze_sentiment)

# -----------------------------
# 5. Aggregate by bank
# -----------------------------
bank_sentiment = df.groupby("bank / app name").agg(
    average_confidence=("confidence", "mean"),
    total_reviews=("review text", "count")
)

print("\nSentiment Summary by Bank:")
print(bank_sentiment)

# -----------------------------
# 6. Aggregate by star rating
# -----------------------------
rating_sentiment = df.groupby("rating (1–5)").agg(
    average_confidence=("confidence", "mean"),
    total_reviews=("review text", "count")
)

print("\nSentiment Summary by Rating:")
print(rating_sentiment)

# -----------------------------
# 7. Save results
# -----------------------------
df.to_csv("reviews_with_sentiment.csv", index=False)

bank_sentiment.to_csv("bank_sentiment_summary.csv")

rating_sentiment.to_csv("rating_sentiment_summary.csv")

print("\nSentiment analysis completed successfully.")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8530.18it/s]



Sentiment Summary by Bank:
                                   average_confidence  total_reviews
bank / app name                                                     
Bank of Abyssinia (BOA)                      0.963556            400
Commercial Bank of Ethiopia (CBE)            0.977700            400
Dashen Bank                                  0.977485            400

Sentiment Summary by Rating:
              average_confidence  total_reviews
rating (1–5)                                   
1                       0.977924            250
2                       0.988783             33
3                       0.972295             57
4                       0.967063             89
5                       0.971331            771

Sentiment analysis completed successfully.
